In [ ]:
"""
RecruiterVisionAI
Skill Recommendation System
Using:
    - Word2Vec Embeddings
    - Random Forest Classifier
    - MultiLabel Skill Prediction

Author: Vidhi
"""

import os
import re
import pickle
import numpy as np
import pandas as pd

from gensim.models import Word2Vec

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    hamming_loss
)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────

JD_PATH = "/home/vidhi/Desktop/Python PGMS/RecruiterVisionAI/dataset/job_title_des.csv"
RESUME_PATH = "/home/vidhi/Desktop/Python PGMS/RecruiterVisionAI/dataset/UpdatedResumeDataSet.csv"

MODEL_PATH = "models/rf_skill_model.pkl"
W2V_PATH = "models/word2vec.model"
MLB_PATH = "models/mlb.pkl"

os.makedirs("models", exist_ok=True)

EMBEDDING_SIZE = 100
WINDOW_SIZE = 5
MIN_COUNT = 2

TEST_SIZE = 0.2
RANDOM_STATE = 42



SKILL_LIBRARY = [
    "python","java","c++","c#","javascript","typescript",
    "react","angular","node.js","django","flask",
    "mysql","mongodb","postgresql","sql",
    "aws","azure","docker","kubernetes",
    "machine learning","deep learning","nlp",
    "tensorflow","keras","pytorch",
    "html","css","bootstrap",
    "git","linux","tableau","power bi",
    "communication","leadership","teamwork",
    "problem solving","project management"
]



def clean_text(text):
    text = str(text).lower()

    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text



def extract_skills(text):

    found = []

    for skill in SKILL_LIBRARY:

        pattern = r'\b' + re.escape(skill) + r'\b'

        if re.search(pattern, text.lower()):
            found.append(skill)

    return list(set(found))

# ─────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────

def load_data():

    print("\nLoading datasets...")

    jd_df = pd.read_csv(JD_PATH)

    jd_df = jd_df.rename(
        columns={"Job Description": "text"}
    )

    resume_df = pd.read_csv(RESUME_PATH)

    resume_col = next(
        (c for c in resume_df.columns if "resume" in c.lower()),
        resume_df.columns[-1]
    )

    resume_df = resume_df.rename(
        columns={resume_col: "text"}
    )

    df = pd.concat([
        jd_df[["text"]],
        resume_df[["text"]]
    ], ignore_index=True)

    df.dropna(inplace=True)

    df["text"] = df["text"].apply(clean_text)

    df["skills"] = df["text"].apply(extract_skills)

    df = df[df["skills"].map(len) > 0]

    print(f"Total valid rows: {len(df)}")

    return df

# ─────────────────────────────────────────────
# TOKENIZATION
# ─────────────────────────────────────────────

def tokenize(text):

    return text.split()

# ─────────────────────────────────────────────
# TRAIN WORD2VEC
# ─────────────────────────────────────────────

def train_word2vec(tokenized_texts):

    print("\nTraining Word2Vec model...")

    model = Word2Vec(
        sentences=tokenized_texts,
        vector_size=EMBEDDING_SIZE,
        window=WINDOW_SIZE,
        min_count=MIN_COUNT,
        workers=4
    )

    return model

# ─────────────────────────────────────────────
# DOCUMENT EMBEDDING
# Average of word vectors
# ─────────────────────────────────────────────

def document_vector(words, model):

    vectors = []

    for word in words:

        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

# ─────────────────────────────────────────────
# BUILD FEATURES
# ─────────────────────────────────────────────

def build_features(df, w2v_model):

    print("\nBuilding embeddings...")

    tokenized = df["text"].apply(tokenize)

    X = np.array([
        document_vector(tokens, w2v_model)
        for tokens in tokenized
    ])

    mlb = MultiLabelBinarizer()

    y = mlb.fit_transform(df["skills"])

    print(f"X Shape: {X.shape}")
    print(f"y Shape: {y.shape}")

    return X, y, mlb



def train_model(X_train, y_train):

    print("\nTraining Random Forest...")

    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model = MultiOutputClassifier(rf)

    model.fit(X_train, y_train)

    return model


def evaluate(model, X_test, y_test):

    print("\nEvaluating model...")

    y_pred = model.predict(X_test)

    print("\n── Metrics ──")

    print(
        f"Accuracy      : "
        f"{accuracy_score(y_test, y_pred):.4f}"
    )

    print(
        f"Hamming Loss  : "
        f"{hamming_loss(y_test, y_pred):.4f}"
    )

    print(
        f"Micro F1      : "
        f"{f1_score(y_test, y_pred, average='micro'):.4f}"
    )

    print(
        f"Macro F1      : "
        f"{f1_score(y_test, y_pred, average='macro'):.4f}"
    )

    print(
        f"Weighted F1   : "
        f"{f1_score(y_test, y_pred, average='weighted'):.4f}"
    )


def predict_skills(text, model, w2v_model, mlb):

    text = clean_text(text)

    tokens = tokenize(text)

    vec = document_vector(tokens, w2v_model)

    vec = vec.reshape(1, -1)

    pred = model.predict(vec)

    skills = mlb.inverse_transform(pred)

    return skills[0]

def main():

    print("\n" + "="*60)
    print("RecruiterVisionAI")
    print("Word2Vec + RandomForest Skill Recommendation")
    print("="*60)

    # Load data
    df = load_data()

    # Tokenization
    tokenized_texts = df["text"].apply(tokenize)

    # Train Word2Vec
    w2v_model = train_word2vec(tokenized_texts)

    # Build features
    X, y, mlb = build_features(df, w2v_model)

    # Train test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE
    )

    print(f"\nTrain Size: {len(X_train)}")
    print(f"Test Size : {len(X_test)}")

    # Train Random Forest
    model = train_model(X_train, y_train)

    # Evaluate
    evaluate(model, X_test, y_test)

    # Save models
    with open(MODEL_PATH, "wb") as f:
        pickle.dump(model, f)

    with open(MLB_PATH, "wb") as f:
        pickle.dump(mlb, f)

    w2v_model.save(W2V_PATH)

    print("\nModels saved successfully!")

    # ─────────────────────────────────────────
    # SAMPLE TEST
    # ─────────────────────────────────────────

    sample_resume = '''
    Experienced Python developer with knowledge
    of Django, AWS, Docker and Machine Learning
    '''

    skills = predict_skills(
        sample_resume,
        model,
        w2v_model,
        mlb
    )

    print("\nPredicted Skills:")
    print(skills)

# ─────────────────────────────────────────────

if __name__ == "__main__":
    main()


RecruiterVisionAI
Word2Vec + RandomForest Skill Recommendation

Loading datasets...
Total valid rows: 2715

Training Word2Vec model...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'



Building embeddings...
X Shape: (2715, 100)
y Shape: (2715, 34)

Train Size: 2172
Test Size : 543

Training Random Forest...

Evaluating model...

── Metrics ──
Accuracy      : 0.3278
Hamming Loss  : 0.0753
Micro F1      : 0.5988
Macro F1      : 0.4854
Weighted F1   : 0.5752

Models saved successfully!

Predicted Skills:
('python',)
